In [2]:
# EVALUATION - PRECISION AT K (FINAL)
# Output: 3 Tabel Rapi sesuai Proposal & Mockup
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

from src.preprocessing.clean_text import clean_text
from src.preprocessing.casefolding import casefolding
from src.preprocessing.tokenizing import tokenizing
from src.preprocessing.stopwords_id import get_stopwords
from src.preprocessing.stemming import stemming


In [3]:
# =========================================================
# LOAD DATA
# =========================================================
base_dir  = os.path.abspath('..')
tfidf_dir = os.path.join(base_dir, 'data', 'tfidf')

with open(os.path.join(tfidf_dir, 'vectorizer.pkl'), 'rb') as f:
    vectorizer = pickle.load(f)

tfidf_matrix = sp.load_npz(os.path.join(tfidf_dir, 'tfidf_matrix.npz'))
doc_index    = pd.read_csv(os.path.join(base_dir, 'data', 'cleaned_papers.csv'))
stop_words   = get_stopwords()

print(f'✅ Total dokumen : {len(doc_index)} artikel')
print(f'✅ Matriks TF-IDF: {tfidf_matrix.shape}')

✅ Total dokumen : 200 artikel
✅ Matriks TF-IDF: (200, 1722)


In [4]:
# =========================================================
# FUNGSI PREPROCESSING QUERY
# Sesuai batasan: stemming hanya untuk bahasa Indonesia
# =========================================================
def preprocess_query(query: str) -> str:
    text   = clean_text(query)
    text   = casefolding(text)
    tokens = tokenizing(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    if all(token.isascii() for token in tokens):
        return ' '.join(tokens)
    tokens = stemming(tokens)
    return ' '.join(tokens)

In [5]:
# =========================================================
# FUNGSI SEARCH
# =========================================================
def search(query: str, top_k: int = 10):
    processed  = preprocess_query(query)
    query_vec  = vectorizer.transform([processed])
    scores     = cosine_similarity(query_vec, tfidf_matrix).flatten()
    ranked_idx = np.argsort(scores)[::-1][:top_k]

    results = doc_index.iloc[ranked_idx][
        ['id', 'title', 'authors', 'year', 'source', 'category']
    ].copy()
    results['similarity_score'] = scores[ranked_idx]
    results['rank']             = range(1, top_k + 1)
    results = results[results['similarity_score'] > 0].reset_index(drop=True)
    results['rank'] = range(1, len(results) + 1)
    return results

print('✅ Fungsi search siap!')

✅ Fungsi search siap!


In [6]:
# =========================================================
# QUERY UJI - 4 KATEGORI SESUAI PROPOSAL
# =========================================================
queries_eval = [
    {"query": "machine learning",    "kategori": "Machine Learning"},
    {"query": "deep learning",       "kategori": "Machine Learning"},
    {"query": "web development",     "kategori": "Web Development"},
    {"query": "web application",     "kategori": "Web Development"},
    {"query": "cyber security",      "kategori": "Cyber Security"},
    {"query": "network security",    "kategori": "Cyber Security"},
    {"query": "mobile application",  "kategori": "Mobile Application"},
    {"query": "android application", "kategori": "Mobile Application"},
    {"query": "data mining",         "kategori": "Machine Learning"},
    {"query": "software security",   "kategori": "Cyber Security"},
]

In [7]:
# =========================================================
# JALANKAN SEARCH SEMUA QUERY
# =========================================================
all_results    = {}
kemunculan_all = defaultdict(int)
kemunculan_kat = defaultdict(lambda: defaultdict(int))

for item in queries_eval:
    q        = item['query']
    kategori = item['kategori']
    results  = search(q, top_k=10)
    all_results[q] = results

    for _, row in results.iterrows():
        kemunculan_all[row['title']] += 1
        kemunculan_kat[kategori][row['title']] += 1

print('✅ Semua query selesai diproses!')


✅ Semua query selesai diproses!


In [8]:
# GROUND TRUTH MANUAL
# 1 = RELEVAN, 0 = TIDAK RELEVAN
# Urutan = rank 1 sampai 10
# ⚠️ WAJIB DIGANTI sesuai penilaianmu sendiri!
# =========================================================
ground_truth = {
    "machine learning":    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # ← GANTI
    "deep learning":       [1, 1, 1, 1, 0, 1, 1, 0, 1, 1],  # ← GANTI
    "web development":     [1, 1, 1, 1, 1, 1, 0, 1, 1, 0],  # ← GANTI
    "web application":     [1, 1, 0, 1, 1, 1, 1, 0, 1, 1],  # ← GANTI
    "cyber security":      [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # ← GANTI
    "network security":    [1, 1, 1, 0, 1, 1, 0, 1, 1, 0],  # ← GANTI
    "mobile application":  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],  # ← GANTI
    "android application": [1, 1, 0, 1, 1, 0, 1, 1, 1, 0],  # ← GANTI
    "data mining":         [1, 1, 1, 0, 1, 1, 0, 1, 0, 1],  # ← GANTI
    "software security":   [1, 1, 1, 1, 0, 1, 1, 0, 1, 1],  # ← GANTI
}

In [9]:
# =========================================================
# TABEL 1 - HASIL PENCARIAN PER QUERY
# (sesuai tampilan mockup sistem)
# =========================================================
print('\n' + '='*75)
print('TABEL 1 - HASIL PENCARIAN PER QUERY')
print('='*75)

tabel1_rows = []

for item in queries_eval:
    q        = item['query']
    kategori = item['kategori']
    results  = all_results[q]
    gt       = ground_truth[q]

    print(f'\n🔍 Query    : "{q}"')
    print(f'📂 Kategori : {kategori}')
    print(f'{"Rank":<5} {"Similarity Score":>16}  {"Penulis":<20}  {"Relevan":>7}  Judul Artikel')
    print('-'*75)

    for i, (_, row) in enumerate(results.iterrows()):
        relevan = '✅ Ya' if i < len(gt) and gt[i] == 1 else '❌ Tidak'
        print(f'{int(row["rank"]):<5} {row["similarity_score"]:>16.4f}  '
              f'{str(row["authors"])[:18]:<20}  {relevan:>7}  '
              f'{str(row["title"])[:35]}')

        tabel1_rows.append({
            'Query'           : q,
            'Kategori'        : kategori,
            'Rank'            : int(row['rank']),
            'Judul'           : row['title'],
            'Penulis'         : row['authors'],
            'Tahun'           : row['year'],
            'Similarity Score': round(row['similarity_score'], 4),
            'Relevan'         : 'Ya' if i < len(gt) and gt[i] == 1 else 'Tidak',
        })



TABEL 1 - HASIL PENCARIAN PER QUERY

🔍 Query    : "machine learning"
📂 Kategori : Machine Learning
Rank  Similarity Score  Penulis               Relevan  Judul Artikel
---------------------------------------------------------------------------
1               0.6232  K Sharifani, M Ami       ✅ Ya  Machine learning and deep learning:
2               0.6099  B Liu, M Ding, S S       ✅ Ya  When machine learning meets privacy
3               0.5976  C Janiesch, P Zsch       ✅ Ya  Machine learning and deep learning:
4               0.5638  AFAH Alnuaimi, THK       ✅ Ya  An overview of machine learning cla
5               0.5608  T Jo                     ✅ Ya  Machine learning foundations
6               0.5598  J Thiyagalingam, M       ✅ Ya  Scientific machine learning benchma
7               0.5578  N Nazareth, YVR Re       ✅ Ya  Financial applications of machine l
8               0.5326  AM Schweidtmann, E       ✅ Ya  Machine learning in chemical engine
9               0.5235  SJ Goodswe

In [10]:
# =========================================================
# TABEL 2 - PRECISION AT K PER QUERY
# =========================================================
K_values  = [5, 10]
eval_rows = []

for item in queries_eval:
    q  = item['query']
    gt = ground_truth[q]
    assert len(gt) == 10, f'Ground truth "{q}" harus 10 angka!'

    row = {'Query': q, 'Kategori': item['kategori']}
    for k in K_values:
        relevan   = sum(gt[:k])
        precision = relevan / k
        row[f'Relevan@{k}'] = relevan
        row[f'P@{k}']       = round(precision, 4)
    eval_rows.append(row)

eval_df = pd.DataFrame(eval_rows)

print('\n\n' + '='*75)
print('TABEL 2 - PRECISION AT K PER QUERY')
print('='*75)
print(eval_df[['Query', 'Kategori', 'Relevan@5', 'P@5',
               'Relevan@10', 'P@10']].to_string(index=False))





TABEL 2 - PRECISION AT K PER QUERY
              Query           Kategori  Relevan@5  P@5  Relevan@10  P@10
   machine learning   Machine Learning          5  1.0          10   1.0
      deep learning   Machine Learning          4  0.8           8   0.8
    web development    Web Development          5  1.0           8   0.8
    web application    Web Development          4  0.8           8   0.8
     cyber security     Cyber Security          5  1.0          10   1.0
   network security     Cyber Security          4  0.8           7   0.7
 mobile application Mobile Application          5  1.0          10   1.0
android application Mobile Application          4  0.8           7   0.7
        data mining   Machine Learning          4  0.8           7   0.7
  software security     Cyber Security          4  0.8           8   0.8


In [11]:
# Rata-rata keseluruhan
print('\n' + '-'*75)
for k in K_values:
    avg = eval_df[f'P@{k}'].mean()
    print(f'Mean Average P@{k}  : {avg:.4f}  ({avg*100:.1f}%)')

# Rata-rata per kategori
print('\nRata-rata P@10 per Kategori:')
kat_avg = eval_df.groupby('Kategori')['P@10'].mean().reset_index()
kat_avg.columns = ['Kategori', 'Rata-rata P@10']
kat_avg['Rata-rata P@10'] = kat_avg['Rata-rata P@10'].round(4)
print(kat_avg.to_string(index=False))



---------------------------------------------------------------------------
Mean Average P@5  : 0.8800  (88.0%)
Mean Average P@10  : 0.8300  (83.0%)

Rata-rata P@10 per Kategori:
          Kategori  Rata-rata P@10
    Cyber Security          0.8333
  Machine Learning          0.8333
Mobile Application          0.8500
   Web Development          0.8000


In [12]:
# =========================================================
# TABEL 3 - JUMLAH KEMUNCULAN ARTIKEL
# =========================================================
print('\n\n' + '='*75)
print('TABEL 3A - JUMLAH KEMUNCULAN ARTIKEL (SEMUA QUERY)')
print('='*75)

kemunculan_df = pd.DataFrame([
    {'Judul Artikel': title[:60], 'Jumlah Kemunculan': count}
    for title, count in sorted(kemunculan_all.items(), key=lambda x: -x[1])
])
print(kemunculan_df.head(10).to_string(index=False))

print('\n' + '='*75)
print('TABEL 3B - JUMLAH KEMUNCULAN PER KATEGORI')
print('='*75)

tabel3b_rows = []
for kat in ['Machine Learning', 'Web Development', 'Cyber Security', 'Mobile Application']:
    if kat not in kemunculan_kat:
        continue
    data = kemunculan_kat[kat]
    top3 = sorted(data.items(), key=lambda x: -x[1])[:3]
    print(f'\n📂 {kat}')
    print(f'  {"Judul Artikel":<55} Kemunculan')
    print(f'  ' + '-'*65)
    for title, count in top3:
        print(f'  {str(title)[:55]:<55} {count}x')
        tabel3b_rows.append({
            'Kategori'        : kat,
            'Judul Artikel'   : title,
            'Jumlah Kemunculan': count
        })




TABEL 3A - JUMLAH KEMUNCULAN ARTIKEL (SEMUA QUERY)
                                               Judul Artikel  Jumlah Kemunculan
Human monkeypox classification from skin lesion images with                   3
A new mobile application of agricultural pests recognition u                  3
Deep learning methods for accurate skin cancer recognition a                  3
Towards a new learning experience through a mobile applicati                  3
User experience analysis on mobile application design using                   3
        A systematic literature review on the cyber security                  3
Analysis of cyber security knowledge gaps based on cyber sec                  3
The difference between cyber security vs information securit                  3
A comprehensive review of cyber security vulnerabilities, th                  3
The role of cyber security in a digitalizing economy: a deve                  3

TABEL 3B - JUMLAH KEMUNCULAN PER KATEGORI

📂 Machine Learning
  Ju

In [13]:
# =========================================================
# SIMPAN SEMUA CSV
# =========================================================
eval_dir = os.path.join(base_dir, 'data', 'evaluation')
os.makedirs(eval_dir, exist_ok=True)

# Tabel 1
pd.DataFrame(tabel1_rows).to_csv(
    os.path.join(eval_dir, 'tabel1_hasil_pencarian.csv'), index=False)

# Tabel 2
eval_df.to_csv(
    os.path.join(eval_dir, 'tabel2_precision_at_k.csv'), index=False)
kat_avg.to_csv(
    os.path.join(eval_dir, 'tabel2_rata_rata_per_kategori.csv'), index=False)

# Tabel 3
kemunculan_df.to_csv(
    os.path.join(eval_dir, 'tabel3a_kemunculan_semua.csv'), index=False)
pd.DataFrame(tabel3b_rows).to_csv(
    os.path.join(eval_dir, 'tabel3b_kemunculan_per_kategori.csv'), index=False)

print('\n\n✅ Semua file tersimpan di: data/evaluation/')
print('   - tabel1_hasil_pencarian.csv')
print('   - tabel2_precision_at_k.csv')
print('   - tabel2_rata_rata_per_kategori.csv')
print('   - tabel3a_kemunculan_semua.csv')
print('   - tabel3b_kemunculan_per_kategori.csv')



✅ Semua file tersimpan di: data/evaluation/
   - tabel1_hasil_pencarian.csv
   - tabel2_precision_at_k.csv
   - tabel2_rata_rata_per_kategori.csv
   - tabel3a_kemunculan_semua.csv
   - tabel3b_kemunculan_per_kategori.csv
